In [ ]:
import os
from google.colab import drive

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
os.listdir('/content/drive/MyDrive/Dataset')

['WIN_20260207_10_59_32_Pro.jpg',
 'acne-open-comedo-59.jpeg',
 'train',
 'Dataset.zip',
 'testing',
 'facial_model.keras']

In [4]:
train_dataset = "/content/drive/MyDrive/Dataset/train"
test_dataset = "/content/drive/MyDrive/Dataset/testing"

In [5]:
os.listdir(train_dataset)

['Wrinkle', 'Rosacea', 'Normal', 'Acne', 'Hyperpigmentation', 'Eczemaa']

In [6]:
os.listdir(test_dataset)

['Acne', 'Eczemaa', 'Hyperpigmentation', 'Normal', 'Rosacea', 'Wrinkle']

In [7]:
train_count = {}
for image in os.listdir(train_dataset):
    image_path = os.path.join(train_dataset,image)
    train_count[image] = len(os.listdir(image_path))
print(train_count)

{'Wrinkle': 1000, 'Rosacea': 1086, 'Normal': 1142, 'Acne': 1092, 'Hyperpigmentation': 1096, 'Eczemaa': 1154}


In [8]:
test_count = {}
for image in os.listdir(test_dataset):
    image_path = os.path.join(test_dataset,image)
    test_count[image] = len(os.listdir(image_path))
print(test_count)

{'Acne': 222, 'Eczemaa': 246, 'Hyperpigmentation': 117, 'Normal': 346, 'Rosacea': 145, 'Wrinkle': 100}


In [9]:
classes = ['Acne', 'Eczemaa', 'Hyperpigmentation', 'Normal', 'Rosacea', 'Wrinkle']

In [10]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

In [11]:
train_data = image_dataset_from_directory(
    train_dataset,
    validation_split = 0.2,
    subset = "training",
    class_names=classes,
    color_mode = "rgb",
    image_size = (224,224),
    shuffle = True,
    seed = 42,
    batch_size = 32
    )

val_data = image_dataset_from_directory(
    train_dataset,
    validation_split = 0.2,
    subset = "validation",
    class_names=classes,
    color_mode = "rgb",
    image_size =(224,224),
    shuffle = False,
    seed = 42,
    batch_size = 32
)

Found 6570 files belonging to 6 classes.
Using 5256 files for training.
Found 6570 files belonging to 6 classes.
Using 1314 files for validation.


In [12]:
test_data = image_dataset_from_directory(
    test_dataset,
    color_mode = "rgb",
    class_names=classes,
    image_size = (224,224),
    shuffle = False,
    seed = 42,
    batch_size = 32
    )

Found 1176 files belonging to 6 classes.


In [13]:
from tensorflow.keras.applications.efficientnet import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

val_data = val_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

test_data = test_data.map(
    lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y),
    num_parallel_calls=AUTOTUNE
)

train_data = train_data.prefetch(AUTOTUNE)
val_data = val_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)

In [14]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential


In [15]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)


base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [16]:
base_model.trainable = False

In [17]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1)
])

In [18]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(6, activation="softmax")
])

In [19]:
from tensorflow.keras.optimizers import Adam
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [22]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2
    )
]

In [23]:
model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks
)

Epoch 1/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 89ms/step - accuracy: 0.9049 - loss: 0.3012 - val_accuracy: 0.9231 - val_loss: 0.2320 - learning_rate: 1.0000e-04
Epoch 2/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 92ms/step - accuracy: 0.9091 - loss: 0.2908 - val_accuracy: 0.9254 - val_loss: 0.2314 - learning_rate: 1.0000e-04
Epoch 3/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 90ms/step - accuracy: 0.9100 - loss: 0.2894 - val_accuracy: 0.9307 - val_loss: 0.2231 - learning_rate: 1.0000e-04
Epoch 4/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 16s 99ms/step - accuracy: 0.9098 - loss: 0.2784 - val_accuracy: 0.9323 - val_loss: 0.2232 - learning_rate: 1.0000e-04
Epoch 5/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 15s 89ms/step - accuracy: 0.9146 - loss: 0.2664 - val_accuracy: 0.9422 - val_loss: 0.2070 - learning_rate: 1.0000e-04
Epoch 6/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 20s 88ms/step - accuracy: 0.9169 - loss: 0.2638 - val_accuracy: 0.9422 - val_loss: 0.2071 - learning_rate: 1.0000e-04
Epoch 7/20
165/165 ━━━━━━━━━━━━━━━━━━━━ 14s 87ms/ste

In [24]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,543,799 (17.33 MB)

 Trainable params: 164,742 (643.52 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

 Optimizer params: 329,486 (1.26 MB)

In [25]:
model.evaluate(test_data)

37/37 ━━━━━━━━━━━━━━━━━━━━ 245s 7s/step - accuracy: 0.7483 - loss: 0.7534


[0.7533671855926514, 0.7482993006706238]

In [26]:
model.save("/content/drive/MyDrive/Dataset/facial_model.keras")

In [27]:
from PIL import Image

In [28]:
image = "/content/drive/MyDrive/Dataset/acne-open-comedo-59.jpeg"

In [29]:
import numpy as np

In [30]:
from tensorflow.keras.applications.efficientnet import preprocess_input

img = Image.open(image)
img = img.convert("RGB")
img = img.resize((224,224))

img = np.array(img).astype(np.float32)

img = preprocess_input(img)

img = np.expand_dims(img, axis=0)


In [31]:
prediction = model.predict(img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 14s 14s/step


In [32]:
for class_name, probability in zip(classes, prediction[0]):
    print(f"{class_name}: {probability*100:.2f}%")

Acne: 12.98%
Eczemaa: 85.71%
Hyperpigmentation: 0.09%
Normal: 0.00%
Rosacea: 1.21%
Wrinkle: 0.01%


In [33]:
import numpy as np

y_true = []

for images, labels in test_data:
    y_true.extend(labels.numpy())

y_true = np.array(y_true)

In [34]:
predictions = model.predict(test_data)

y_pred = np.argmax(predictions, axis=1)

37/37 ━━━━━━━━━━━━━━━━━━━━ 17s 288ms/step


In [35]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

print(cm)

[[124  78   7   1   7   5]
 [  2 235   6   1   2   0]
 [ 10  19  54  28   3   3]
 [  5   9  22 297   2  11]
 [  5  54   0   0  86   0]
 [  1   1   2  10   2  84]]


In [36]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_true,
        y_pred,
        target_names=classes
    )
)

                   precision    recall  f1-score   support

             Acne       0.84      0.56      0.67       222
          Eczemaa       0.59      0.96      0.73       246
Hyperpigmentation       0.59      0.46      0.52       117
           Normal       0.88      0.86      0.87       346
          Rosacea       0.84      0.59      0.70       145
          Wrinkle       0.82      0.84      0.83       100

         accuracy                           0.75      1176
        macro avg       0.76      0.71      0.72      1176
     weighted avg       0.78      0.75      0.74      1176

